# 05 – Evaluation (tasks 24, 26, 27)

The last step, in this order:
1. **Choose the final model** (task 26): four candidates compete on five different splits of the cross-validation data
   (README decisions 29–32); the final choice was made in the coaching on 25 Nov 2026 (decision 33).
   The test set is not used here.
2. **Error analysis** of the final model on its cross-validation predictions (task 24): where and how does it go wrong?
   It does not choose the model; it shows the limits of the model we hand in.
3. **Test set:** the final model is evaluated **exactly once** on the test set (task 26). Houses first, all buildings beside them.
4. **Hit-rate curve** on the test houses for several list lengths, with the number of owners each length means (task 27).

The model race in section 1 takes about 5 minutes.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge

from config import SEED, THRESHOLD, TABLES, FIGURES, load_cv, load_test
from metrics import evaluate, _top_positions   # _top_positions: the same ranking (and tie-break) as the selection measure
from pipeline import split_xy, build_pipeline, ExpertRuleBaseline
from cv import (make_folds, run_variant, race_table, corrected_difference, choose_model, plot_race,
                best_alpha, SELECTION)

MIN_N = 20   # error analysis: groups with fewer houses are shown but not interpreted

## 1. Choose the final model (task 26)

**Question:** which model goes to the test set? In `04_models.ipynb` several variants lie within the noise of one split
(standard error about 3 points; with about 47 houses in the top 20% of a fold, one house is about 2 points). One split can
pick the winner by chance, and the more variants we compare, the more optimistic the best value is. So a small list of
candidates, fixed in the README **before** this run, competes on **five different splits** of the cross-validation data.

**Candidates** (from simple to complex, decision 29):
1. Ridge trained on houses only
2. Ridge trained on houses only, with compactness (1 / floors, 1 / √floor area)
3. Ridge with house interactions and compactness, trained on all buildings
4. Gradient boosting trained on all buildings (settings as in `04_models.ipynb`)

The two baselines (expert rule, training mean) run along as references, not as candidates.

**Five runs** (decision 30): run 1 uses the folds of `cv.csv` (the folds of `04_models.ipynb`); runs 2–5 split the same
12,710 buildings again into 5 folds with other seeds, with the same rules (groups together, same share of houses).
Every candidate is tuned again in every run (alpha or number of leaves, decision 18). The test set is not used.

**Rule** (decision 32): the leader has the highest mean selection measure over all 25 folds. Every other candidate is
compared with the leader fold by fold on the same houses. The runs reuse the same houses, so the 25 values are not
independent; we use the **corrected standard error** (Nadeau and Bengio): standard deviation × √(1/25 + 1/4).
If a simpler candidate is at most one corrected standard error behind the leader, the simplest such candidate wins.
**Final choice** (decision 33): made in the coaching, see 1g.

### 1a. Candidates and runs

Settings as in `04_models.ipynb` (section 2): the same values of alpha and the same tree settings.

In [ ]:
RIDGE_ALPHAS = np.logspace(-2, 5, 29)   # 0.01 to 100,000, as in 04
LEAF_VALUES = [7, 15, 31]               # tree size, as in 04
TREE_KWARGS = {"learning_rate": 0.05, "max_iter": 300, "early_stopping": False, "random_state": SEED}

CANDIDATES = {   # from simple to complex (decision 32)
    "Ridge, houses only": {"model": Ridge, "tune": ("alpha", RIDGE_ALPHAS), "train_on": "houses"},
    "Ridge, houses only, compactness": {"model": Ridge, "tune": ("alpha", RIDGE_ALPHAS), "train_on": "houses",
                                        "switches": {"compactness": True}},
    "Ridge, house interactions, compactness": {"model": Ridge, "tune": ("alpha", RIDGE_ALPHAS),
                                               "switches": {"house_interactions": True, "compactness": True}},
    "Gradient boosting, reference": {"model": HistGradientBoostingRegressor, "tune": ("max_leaf_nodes", LEAF_VALUES),
                                     "kwargs": TREE_KWARGS},
}
REFERENCES = {   # baselines, not candidates
    "Expert rule, baseline": {"model": ExpertRuleBaseline, "baseline": True},
    "Training mean, baseline": {"model": DummyRegressor, "baseline": True},
}
SEEDS = {1: None, 2: SEED + 1, 3: SEED + 2, 4: SEED + 3, 5: SEED + 4}   # run 1: the folds of cv.csv

train = load_cv()
X, y, is_house, groups = split_xy(train)
print("Buildings:", len(train), "| houses:", is_house.sum())

### 1b. Five splits of the cross-validation data

Run 1 is the split everyone has used so far. For runs 2–5 the same buildings are split again. The table checks every
split: houses per fold, share of houses above 450 per fold (not balanced on purpose, as in decision 9), and groups
split across folds (must be 0).

In [ ]:
folds_by_run = {run: train["fold"].to_numpy() if seed is None else make_folds(train, seed)
                for run, seed in SEEDS.items()}
print("make_folds with the project seed gives the folds of cv.csv:",
      bool((make_folds(train, SEED) == folds_by_run[1]).all()))

rows = []
for run, folds in folds_by_run.items():
    split = train.assign(fold=folds)
    houses = split[split["house"]]
    rows.append({
        "run": run, "seed": SEED if SEEDS[run] is None else SEEDS[run],
        "houses per fold": "–".join(str(n) for n in houses.groupby("fold").size()),
        "houses above 450 per fold (%)": "–".join(f"{v:.0f}" for v in houses.groupby("fold")["y"].apply(lambda s: (s > THRESHOLD).mean() * 100)),
        "groups split across folds": int((split.groupby("group_id")["fold"].nunique() > 1).sum()),
        "same fold as in run 1 (%)": round((folds == folds_by_run[1]).mean() * 100, 1),
    })
pd.DataFrame(rows).set_index("run")

**Reading the table.** By chance about 20% of the buildings land in the same fold number as in run 1; the folds are
really new. Every run keeps the groups together.

### 1c. The race

Every candidate and baseline through every run: tune, then cross-validate with the chosen setting. Takes a few minutes.

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning)
results = {name: {} for name in {**CANDIDATES, **REFERENCES}}
for run, folds in folds_by_run.items():
    for name, spec in {**CANDIDATES, **REFERENCES}.items():
        r = run_variant(spec, X, y, is_house, folds)
        r["per_fold"].insert(0, "run", run)
        if r["tuning"] is not None:
            r["tuning"].insert(0, "run", run)
        results[name][run] = r
    print(f"run {run}:", " | ".join(f"{name}: {results[name][run]['value']:.3g}" for name in CANDIDATES
                                    if not np.isnan(results[name][run]["value"])))

per_fold = {name: pd.concat([results[name][run]["per_fold"] for run in SEEDS], ignore_index=True) for name in results}

### 1d. Check

 run 1 uses the same folds and settings as `04_models.ipynb`, so it must give the same numbers.

In [ ]:
summary_04 = pd.read_csv(TABLES / "models_cv_summary.csv").query("group == 'houses'").set_index("variant")
for name in results:
    run1 = per_fold[name].query("run == 1 and group == 'houses'")[SELECTION].mean()
    print(f"{name:<42} run 1: {run1 * 100:5.1f}% | 04_models: {summary_04.loc[name, SELECTION + '_mean'] * 100:5.1f}%")

### 1e. Results

Mean over all 25 folds with its corrected standard error, the mean of every run, and the other measures on houses.

In [ ]:
race = race_table(per_fold)
race

**Fold by fold against the leader.** Positive = better than the leader. `better on average in` counts the runs in which
the candidate is ahead on average.

In [ ]:
order = list(CANDIDATES)
means = race.loc[order, "top 20%: above 450 (%)"]
leader = means.idxmax()
differences = pd.DataFrame([{"variant": name, "compared with": leader,
                             **corrected_difference(per_fold[name], per_fold[leader])[1]}
                            for name in results if name != leader]).set_index("variant")
differences

### 1f. Figure

Small dots: mean of one run. Large dot: mean over all runs ± corrected standard error. Grey lines: references.

In [ ]:
random_share = (y[is_house] > THRESHOLD).mean() * 100
reference = {f"random choice ({random_share:.1f}%)": random_share,
             f"expert rule ({race.loc['Expert rule, baseline', 'top 20%: above 450 (%)']:.1f}%)":
                 race.loc["Expert rule, baseline", "top 20%: above 450 (%)"]}
plot_race({name: per_fold[name] for name in CANDIDATES}, reference=reference, path=FIGURES / "choice_race.png")
plt.show()

### 1g. The decision

**By the rule of decision 32:** the simplest candidate within one corrected standard error of the leader.

**In the coaching (decision 33)** we chose ridge on houses only **with compactness**, following the lecturer: preferring the
simpler model pays off when it saves something, for example a data source. Compactness needs no extra data (only floor area
and floors, which the model uses anyway), has no drawback, has a physical reason and is ahead in 4 of 5 runs.

In [ ]:
winner, leader, decision = choose_model({name: per_fold[name] for name in CANDIDATES}, order)
display(decision)
print("Leader (highest mean):", leader)
print("Winner by the rule (simplest candidate within one corrected SE of the leader):", winner)

FINAL = "Ridge, houses only, compactness"   # chosen in the coaching on 25 Nov 2026 (README decision 33)
print("Final model (coaching):", FINAL)

### 1h. Would a combination of the candidates be better? (stacking)

The lecturer suggested a meta-model that combines the candidates. Combining only helps if the candidates make **different**
errors: then their mistakes partly cancel out. We check this in two ways, on all five runs:
- **Correlation of the errors** (prediction − true value) of the four candidates on the houses. Close to 1 = they go wrong on
  the same houses, and combining cannot help much.
- **A simple average** of the predictions, scored like every candidate (per run and fold, houses of the validation fold).
  It needs no fitted weights, so it cannot be optimistic. If even the average is not better, a fitted meta-model has little room.

In [ ]:
names = list(CANDIDATES)
errors = pd.DataFrame({name: np.concatenate([results[name][run]["oof"][is_house] - y[is_house] for run in SEEDS])
                       for name in names})
print("Correlation of the errors on houses (all five runs):")
display(errors.corr().round(2))

def score_average(members):
    # the average prediction of some candidates, scored per run and fold like every variant
    rows = []
    for run, folds in folds_by_run.items():
        average = np.mean([results[m][run]["oof"] for m in members], axis=0)
        for k in np.unique(folds):
            val = folds == k
            r = evaluate(y[val], average[val], is_house[val])
            r.insert(0, "fold", k)
            r.insert(0, "run", run)
            rows.append(r)
    return pd.concat(rows, ignore_index=True)

averages = {"Average: ridge with compactness and gradient boosting": score_average([FINAL, "Gradient boosting, reference"]),
            "Average of all four candidates": score_average(names)}
display(race_table({FINAL: per_fold[FINAL], **averages}))
stacking = pd.DataFrame([{"variant": name, "compared with": FINAL, **corrected_difference(pf, per_fold[FINAL])[1]}
                         for name, pf in averages.items()]).set_index("variant")
stacking

### 1i. Setting of the final model

The tuning curves of the five runs are averaged; the value with the highest mean over all 25 folds wins (decision 18,
MAE on houses as tie-break).

In [ ]:
param = CANDIDATES[FINAL]["tune"][0]
tuning = pd.concat([results[FINAL][run]["tuning"] for run in SEEDS], ignore_index=True)
measures = [c for c in tuning.columns if c.endswith("_mean")]
tuning_mean = tuning.groupby([param, "group"], as_index=False)[measures].mean()
final_value = best_alpha(tuning_mean, param=param)

per_run = {run: results[FINAL][run]["value"] for run in SEEDS}
print("Chosen in each run:", {run: round(v, 3) for run, v in per_run.items()})
print(f"Final model: {FINAL} | {param} = {final_value:.4g}")
curve = tuning_mean.query("group == 'houses'")[[param, SELECTION + "_mean", "mae_mean"]].copy()
curve[SELECTION + "_mean"] = (curve[SELECTION + "_mean"] * 100).round(1)
curve.round(2).set_index(param).T

### 1j. Save

- `output/tables/choice_per_fold.csv`: every candidate and baseline, every run and fold
- `output/tables/choice_summary.csv`: the results table of 1e
- `output/tables/choice_differences.csv`: every variant against the leader (1e)
- `output/tables/choice_decision.csv`: the decision table of 1g
- `output/tables/choice_stacking.csv`: the simple averages against the final model (1h)
- `output/tables/choice_final_model.csv`: the final model and its setting, read in section 3
- `output/figures/choice_race.png`

In [ ]:
pd.concat(per_fold, names=["variant"]).reset_index(level=0).to_csv(TABLES / "choice_per_fold.csv", index=False)
race.to_csv(TABLES / "choice_summary.csv")
differences.to_csv(TABLES / "choice_differences.csv")
decision.to_csv(TABLES / "choice_decision.csv")
stacking.to_csv(TABLES / "choice_stacking.csv")
spec = CANDIDATES[FINAL]
pd.DataFrame([{"variant": FINAL, "winner by the rule": winner, "model": spec["model"].__name__,
               "trained on": spec.get("train_on", "all"), "switches": json.dumps(spec.get("switches", {})),
               "fixed arguments": json.dumps(spec.get("kwargs", {})), "setting": param, "value": final_value}]
             ).to_csv(TABLES / "choice_final_model.csv", index=False)
print("Saved the choice tables; final model:", FINAL, f"({param} = {final_value:.4g})")

### 1k. What we found

- **All four candidates lie within one corrected standard error of each other** (58.1% to 60.1%, corrected SE 4–5 points);
  all beat the expert rule (44.7%) and random choice (38%) clearly, in every run.
- **Leader: ridge on houses only with compactness, 60.1%**, also the most stable candidate (58.7% to 61.3% over the five runs).
  Ridge without compactness is 1.6 ± 1.7 points behind: worse in 14 and better in 4 of 25 folds, ahead on average in 1 of 5 runs.
- **Decision.** The rule of decision 32 picks the simpler ridge without compactness, because the lead is within one corrected SE.
  In the coaching we chose compactness (decision 33): it needs no extra data, has no drawback, has a physical reason and is
  ahead in 4 of 5 runs.
- **One split can mislead.** Gradient boosting had the best value on the split of `04_models.ipynb` (61.7%), but 58.8% on
  average over the five splits.
- **No meta-model (stacking).** The errors of the candidates are very similar (correlation 0.94 to 0.99): they go wrong on
  the same houses. A simple average of their predictions does not rank houses better than the final model alone
  (59.4% and 58.9% vs. 60.1%; −0.7 ± 2.2 and −1.2 ± 1.8 points).
- **Setting.** alpha = 3.16, the best mean over all 25 folds (59.7% on the averaged tuning curve; chosen per run between
  0.56 and 10). The race means are slightly optimistic because alpha is chosen on the same folds it is scored on
  (60.1% with alpha tuned per run vs. 59.7% with one alpha for all runs); the unbiased number comes from the test set (section 3).

## 2. Error analysis (task 24)

**Question:** where does our model go wrong? We already have one prediction per building from `04_models.ipynb`, made by a
model that never saw that building (out-of-fold). We do not train anything new; we only look at the errors.

Three parts, houses first (they are what the energy office cares about), all buildings next to them:
1. **Plausibility:** are the predictions sensible at all (no negative values, nothing absurd)?
2. **Errors by group:** construction period, energy group, building class. Is the model too high or too low anywhere?
3. **The office's list:** which houses end up as false alarms in the top 20%, and which problem houses does the list miss?

**Error = prediction − true value.** Positive = the model guessed too high, negative = too low.

Input: `output/tables/models_cv_oof.csv` (from `04_models.ipynb`). The test set is not used.
Output: `output/tables/evaluation_*.csv` and `output/figures/evaluation_*.png`.

### 2a. Load predictions and building characteristics

In [ ]:
MODEL = FINAL   # the final model of section 1: a column of output/tables/models_cv_oof.csv
oof = pd.read_csv(TABLES / "models_cv_oof.csv")
info = load_cv()[
    ["EGID", "construction_period", "energy_group", "building_class", "heating_system", "floor_area", "dwellings"]]
df = oof.merge(info, on="EGID", how="left", validate="one_to_one")

df["pred"] = df[MODEL]
df["error"] = df["pred"] - df["y"]           # positive = predicted too high
df["abs_error"] = df["error"].abs()

houses = df[df["house"]].copy()
print("Model analysed:", MODEL)
print("Buildings:", len(df), "| houses:", len(houses))
assert df["pred"].notna().all(), "a building has no prediction"

### 2b. Plausibility: are the predictions sensible?

The heating index of our buildings lies between 50 and 1,500 (cleaning rule). A prediction outside that range,
or a negative one, would be impossible. We also compare the spread: linear models usually predict
**too close to the average** (they "play it safe"), which we check by comparing the standard deviations.

In [ ]:
def describe(d):
    return pd.DataFrame({
        "true value": d["y"].describe(),
        "prediction": d["pred"].describe(),
    }).round(0)

print("Houses")
print(describe(houses).to_string())
print()
print("All buildings")
print(describe(df).to_string())
print()
print("Predictions below 50:", (df["pred"] < 50).sum(), "| below 0:", (df["pred"] < 0).sum(),
      "| above 1,500:", (df["pred"] > 1500).sum())
print("Spread of predictions vs. true values (houses):",
      round(houses["pred"].std()), "vs.", round(houses["y"].std()))

**Predicted vs. true value, houses.** Every dot is a house. On the diagonal the prediction is perfect.
The dashed lines are the threshold of 450: the top-right box are problem houses the model also sees as high.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(houses["y"], houses["pred"], s=10, alpha=0.4, color="C0")
lims = [0, max(houses["y"].max(), houses["pred"].max()) * 1.05]
ax.plot(lims, lims, color="black", linewidth=1, label="perfect prediction")
ax.axvline(THRESHOLD, color="grey", linestyle="--", linewidth=1, label="threshold (450)")
ax.axhline(THRESHOLD, color="grey", linestyle="--", linewidth=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("True heating index (MJ/m² per year)")
ax.set_ylabel("Predicted heating index (MJ/m² per year)")
ax.set_title(f"Houses: predicted vs. true ({MODEL})")
ax.legend(loc="upper left")
fig.tight_layout()
fig.savefig(FIGURES / "evaluation_predicted_vs_true.png", dpi=150)
plt.show()

**Error by true value.** We sort the houses into bands of their true value and look at the mean error in each band.
If the model plays it safe, it guesses too high for good houses (positive bars on the left) and too low for bad
houses (negative bars on the right).

In [ ]:
boundaries = [0, 250, 350, 450, 550, 700, 1500]  # groups of boundaries
def true_band(value):
    if value <= 250:
        return "1: up to 250"
    if value <= 350:
        return "2: 250–350"
    if value <= 450:
        return "3: 350–450"
    if value <= 550:
        return "4: 450–550"
    if value <= 700:
        return "5: 550–700"
    return "6: above 700"

houses["true band"] = houses["y"].map(true_band) #map the value of y to the bands in the define function
by_band = houses.groupby("true band").agg( #calculates three things per boundary
    count=("y", "size"), mean_error=("error", "mean"), mae=("abs_error", "mean")).round(1) #calculates 1: number of values in that boundary, 2: average error, 3: average absolute error
print(by_band.to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(by_band.index.astype(str), by_band["mean_error"], color="C0")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_xlabel("True heating index of the house (band)")
ax.set_ylabel("Mean error (prediction − true)")
ax.set_title("Houses: the model is too high for good houses and too low for bad ones")
fig.tight_layout()
fig.savefig(FIGURES / "evaluation_error_by_true_value.png", dpi=150)
plt.show()

### 2c. Errors by group

One function makes the same table for any characteristic:
- **n:** number of buildings in the group (small groups are noisy, look at n first)
- **mean error:** bias. Clearly above 0 = the model is systematically too high for this group, below 0 = too low
- **MAE:** how far off a typical building is
- **above 450 true / predicted (%):** how often the group really is above 450 vs. how often the model says so

In [ ]:
def error_table(d, by): 
    table = d.groupby(by).agg( # puts houses into piles by period, energy group etc. and calculates the following metrics for each pile
        n=("y", "size"), # how many houses in the pile
        mean_true=("y", "mean"), # average true value of each  pile
        mean_pred=("pred", "mean"), # average predicted value of each pile
        mean_error=("error", "mean"), # average error of each pile
        mae=("abs_error", "mean"), # average absolute error of each pile
        above_450_true=("y", lambda s: (s > THRESHOLD).mean() * 100), #what percentage of houses in the pile are actually above 450
        above_450_pred=("pred", lambda s: (s > THRESHOLD).mean() * 100), #what percentage of houses in the pile are predicted to be above 450
    ).round(1) #round to 1 decimal place
    table["small group"] = np.where(table["n"] < MIN_N, "yes", "") # creates a column called "small group" and mark groups with fewer than 20 houses as "yes"
    return table

**Construction period (houses).**

Construction periods in the register are in chronological order except "Vor 1919" (before 1919), which we put first.

In [ ]:
period_order = ["Vor 1919", "1919-1945", "1946-1960", "1961-1970", "1971-1980", "1981-1985", "1986-1990",
                "1991-1995", "1996-2000", "2001-2005", "2006-2010", "2011-2015", "Nach 2015", "unknown"] # order of construction periods for display

by_period = error_table(houses, "construction_period") # create a table of error metrics by construction period
by_period = by_period.reindex([p for p in period_order if p in by_period.index]) # reorders the rows of the table if theres any construction periods missing from the data
by_period


**Result (houses, by construction period).**

- **On average the model is right in almost every period** (mean error between −18 and +17), because the construction
  period is a feature. 2006–2010 is predicted somewhat too low (−18); 2011–2015 (+59) has only 16 houses, too few to interpret.
- **But it flags the wrong periods.** It puts far more houses from 1946–1980 above 450 than really are
  (1946–1960: 74.3% predicted vs. 41.9% true; 1961–1970: 86.0% vs. 61.7%; 1971–1980: 71.9% vs. 42.5%), and far fewer
  from 1986–2010 (1991–1995: 19.7% vs. 42.3%; 2006–2010: 8.7% vs. 33.7%). The model plays it safe (section 2b): houses
  of an "old" period all get a high value, houses of a "new" period a low one, although both periods have problem houses.
- **Largest errors in 1946–1970** (MAE 150 and 130), smallest in 1986–1990 (75).

**Energy group (houses).**

In [ ]:
by_energy = error_table(houses, "energy_group").sort_values("n", ascending=False) # create a table of error metrics by energy group and sort it by number of houses in each group
by_energy


**Result (houses, by energy group).** For gas and oil the model is right on average (mean error −2.5 and +2.4), but it
flags far too many oil houses (72.1% predicted above 450 vs. 45.3% true) and too few gas houses (34.7% vs. 41.2%).
Heat pumps are predicted too high (+28), but no heat-pump house is predicted above 450, so they do not reach the list.
District heat (15 houses) and electric (14) are too small to interpret.

**Building class (all buildings).**

Houses only have two classes (1 or 2 dwellings), so here we look at all buildings. Only classes with at least 50 buildings.
The final model is trained on houses only and is **not meant** for other buildings; for them the table only shows
what would happen if it were used there anyway.

In [ ]:
by_class = error_table(df, "building_class").query("n >= 50").sort_values("n", ascending=False) # create a table of error metrics by building class, filter out groups with fewer than 50 houses, and sort it by number of houses in each group
by_class

**Result (all buildings, by building class).** For the two house classes the model is right on average (1 dwelling: 0.0,
2 dwellings: +1.9). For other building types it is far off, mostly too low (hospitals −174, silos and storage −107,
other buildings −101): they use energy in ways houses do not. This confirms that the model must only be used for houses,
which is what the office needs.

### 2d. The office's list: hits, false alarms and missed houses

The selection measure takes, in each validation fold, the 20% of houses with the highest prediction. These are the
owners the office would contact. Every house falls into one of four boxes:

| | in the top 20% (contacted) | not contacted |
|---|---|---|
| **really above 450** | hit | missed |
| **really below 450** | false alarm | correctly left out |

We rebuild exactly this list (same ranking and tie-break as in `metrics.py`) and ask: what do false alarms and
missed houses look like?

In [ ]:
houses["contacted"] = False
for k in houses["fold"].unique():
    in_fold = houses.index[houses["fold"] == k]                       # houses of validation fold k
    top = _top_positions(houses.loc[in_fold, "pred"].to_numpy(), 0.20)  # the same list as the selection measure
    houses.loc[in_fold[top], "contacted"] = True

above = houses["y"] > THRESHOLD
houses["outcome"] = np.select(
    [houses["contacted"] & above, houses["contacted"] & ~above, ~houses["contacted"] & above],
    ["hit", "false alarm", "missed"], default="correctly left out")

counts = houses["outcome"].value_counts()
print(counts.to_string())
print()
print("Share of hits among contacted houses:", round(counts["hit"] / houses["contacted"].sum() * 100, 1), "%",
      "(the same as run 1 of the final model in section 1)")

**How far off are the false alarms?** A false alarm at 430 is almost a hit; a false alarm at 250 is a real mistake.

In [ ]:
print("True value of false alarms:")
print(houses.loc[houses["outcome"] == "false alarm", "y"].describe().round(0).to_string())
print()
print("False alarms just below the threshold (400–450):",
      ((houses["outcome"] == "false alarm") & (houses["y"] >= 400)).sum(),
      "of", (houses["outcome"] == "false alarm").sum())
print()
print("True and predicted value of missed problem houses:")
print(houses.loc[houses["outcome"] == "missed", ["y", "pred"]].describe().round(0).to_string())

**Which houses are they?** Share of each construction period and energy group within each outcome.
If, for example, almost all false alarms are old oil-heated houses, the model relies too much on that combination.

In [ ]:
def share_by_outcome(column):
    table = pd.crosstab(houses[column], houses["outcome"], normalize="columns").mul(100).round(1)
    return table[["hit", "false alarm", "missed", "correctly left out"]]

outcome_by_period = share_by_outcome("construction_period")
outcome_by_period = outcome_by_period.reindex([p for p in period_order if p in outcome_by_period.index])
print("Construction period (% of each column)")
print(outcome_by_period.to_string())
print()
print("Energy group (% of each column)")
print(share_by_outcome("energy_group").to_string())

### 2e. The largest errors

The 10 houses with the largest error in each direction. Do they look like data errors (e.g. a very small floor area,
an implausible value) or like real but unusual houses?

In [ ]:
show_cols = ["EGID", "y", "pred", "error", "construction_period", "energy_group", "heating_system", "floor_area", "dwellings"]
print("Predicted much too LOW (problem houses the model does not see):")
print(houses.nsmallest(10, "error")[show_cols].round(0).to_string(index=False))
print()
print("Predicted much too HIGH:")
print(houses.nlargest(10, "error")[show_cols].round(0).to_string(index=False))

### 2f. Save

In [ ]:
by_period.to_csv(TABLES / "evaluation_errors_by_period_houses.csv")
by_energy.to_csv(TABLES / "evaluation_errors_by_energy_houses.csv")
by_class.to_csv(TABLES / "evaluation_errors_by_class_all.csv")
by_band.to_csv(TABLES / "evaluation_errors_by_true_value_houses.csv")
houses[["EGID", "fold", "y", "pred", "error", "contacted", "outcome"]].to_csv(TABLES / "evaluation_house_outcomes.csv", index=False)
print("Saved the evaluation tables and figures for", MODEL)

### 2g. What we found (final model: ridge on houses only with compactness)

- **Plausible.** For houses the predictions lie between 160 and 608 (true values 74 to 1,499); none is negative. The three
  predictions below 50 are other buildings, for which the model is not meant.
- **The model plays it safe.** The predictions spread only half as much as the true values (standard deviation 80 vs. 152).
  Good houses are predicted far too high (+168 for houses up to 250) and very bad houses far too low (−314 above 700).
  Around the threshold (350–550) the errors are smaller (MAE about 65). This explains the large MAE (about 106) while the
  ranking still works: the order is mostly right, the extremes are pulled towards the middle.
- **The list is driven by oil heating and age.** About a third of houses heat with oil (380 of 1,171), but oil houses make up
  70% of the hits and 74% of the false alarms; all houses on the list but one were built before 1986. The model in effect says
  "old and oil-heated → contact".
- **Missed problem houses** are mostly heated with gas (67%) and often built before 1919 (21%) or in 2006–2010 (12%).
  Their median prediction is 434, their median true value 549: they sit just below the cut.
- **Many false alarms are near misses.** 56 of 93 false alarms lie between 400 and 450 (median 417); for the office these
  owners are close to the threshold anyway.
- **The largest errors** are single houses with extreme values (e.g. 1,499 predicted as 416, or 91 predicted as 533).
  Some may be data errors or recent renovations that the register does not show; the model cannot know either.

## 3. Test set (task 26)

The final model of section 1 (read from `choice_final_model.csv`, so the setting cannot be typed wrongly) is trained once on
the cross-validation data (the houses, as chosen) and evaluated **exactly once** on the test set. The expert rule is trained
the same way as in `04_models.ipynb` and shown beside it as the bar the model has to clear. Only houses are reported: the
model is trained on houses and meant for houses. Nothing is changed after looking at the result.

In [ ]:
final = pd.read_csv(TABLES / "choice_final_model.csv").iloc[0]   # written in section 1
assert final["variant"] == FINAL and final["model"] == "Ridge"
switches, alpha = json.loads(final["switches"]), float(final["value"])
print("Final model:", FINAL, "| alpha =", round(alpha, 3), "| switches:", switches, "| trained on:", final["trained on"])

cv_data = load_cv()
X_cv, y_cv, house_cv, _ = split_xy(cv_data)
train_rows = house_cv if final["trained on"] == "houses" else np.ones(len(y_cv), bool)
final_model = build_pipeline(Ridge(alpha=alpha), **switches).fit(X_cv[train_rows], y_cv[train_rows])
expert_rule = ExpertRuleBaseline().fit(X_cv, y_cv)   # baseline for comparison, trained like in 04
columns = final_model.named_steps["prep"].get_feature_names_out()
print("Trained on", train_rows.sum(), "buildings | columns:", len(columns),
      "| compactness columns:", [c for c in columns if c.startswith("inv_")])

In [ ]:
test = load_test()   # the only place in the project where the test set is loaded
X_test, y_test, house_test, _ = split_xy(test)
print("Test set:", len(test), "buildings,", house_test.sum(), "houses")

pred_test = final_model.predict(X_test)
rows = []
for name, score in [(f"{FINAL} (final)", pred_test),
                    ("Expert rule (baseline)", expert_rule.predict(X_test))]:
    r = evaluate(y_test, score, house_test)
    r.insert(0, "model", name)
    rows.append(r)
test_result = pd.concat(rows, ignore_index=True)
test_result = test_result[test_result["group"] == "houses"].reset_index(drop=True)   # the model is meant for houses only

print("Random choice:", round((y_test[house_test] > THRESHOLD).mean() * 100, 1), "% of test houses are above", THRESHOLD)
test_result.round(3)

In [ ]:
test_result.to_csv(TABLES / "evaluation_test_result.csv", index=False)
pd.DataFrame({"EGID": test["EGID"], "house": test["house"], "y": test["y"],
              "pred": pred_test, "pred_expert_rule": expert_rule.predict(X_test)}
             ).to_csv(TABLES / "evaluation_test_predictions.csv", index=False)
print("Saved evaluation_test_result.csv and evaluation_test_predictions.csv")

**Result (test set, evaluated once).**

- **Houses, top 20% (39 of 195 test houses):** 61.5% are really above 450 (24 houses), against 35.9% by random choice.
  This matches the cross-validation (60.1%): the model works on houses it has never seen.
- **Against the expert rule the test set is not decisive.** The rule reaches 59.0% (23 houses): the model is one house ahead
  in the top 20% and one house behind in the top 10% (55.0% vs. 60.0%, 20 houses). With 39 houses on the list, one house is
  2.6 points, and the standard error of a share near 60% is about 8 points (√(0.6 · 0.4 / 39)). The rule did unusually well on
  these test houses (59.0% vs. 44.7% in cross-validation). The more reliable comparison is the cross-validation over 1,171
  houses and 25 folds: +15.4 ± 4.6 points for the model, better in 24 of 25 folds.
- **MAE on houses:** 105.2 for the model vs. 109.7 for the rule.


## 4. Hit-rate curve (task 27)

The selection measure looks at one list length (top 20%). The curve shows **every** list length on the test houses:
- x-axis: share of houses the office contacts (1% to 100% of the ranking), with a second scale for the number of
  owners this means in the target group (task 13);
- y-axis: share of the contacted houses that are really above 450;
- references: random choice (flat at the share of houses above 450) and a perfect model.

In [ ]:
# task 13: undeclared houses in the target group = last step of the table written by 03_describe.ipynb (21,377)
TARGET_GROUP_HOUSES = int(pd.read_csv(TABLES / "describe_target_group_steps.csv")["buildings"].iloc[-1])

pred = pd.read_csv(TABLES / "evaluation_test_predictions.csv")
houses_test = pred[pred["house"]].reset_index(drop=True)
above = (houses_test["y"] > THRESHOLD).to_numpy()
n = len(houses_test)
base_rate = above.mean()   # random choice: share of test houses above 450

def hit_rate_curve(score):
    # for every list length k: share of the top k houses that are really above 450
    # (same ranking and tie-break as the selection measure in metrics.py)
    order = _top_positions(score.to_numpy(), 1.0)
    hits = np.cumsum(above[order])
    k = np.arange(1, n + 1)
    return k / n * 100, hits / k * 100

x, model_curve = hit_rate_curve(houses_test["pred"])
_, rule_curve = hit_rate_curve(houses_test["pred_expert_rule"])
perfect_curve = np.minimum(1, above.sum() / np.arange(1, n + 1)) * 100   # all problem houses first

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, model_curve, color="C0", linewidth=2, label=f"{FINAL} (final model)")
ax.plot(x, rule_curve, color="C1", linewidth=1.2, label="Expert rule (baseline)")
ax.plot(x, perfect_curve, color="black", linestyle=":", linewidth=1.2, label="Perfect model")
ax.axhline(base_rate * 100, color="grey", linestyle="--", linewidth=1.2,
           label=f"Random choice ({base_rate * 100:.1f}%)")
for top in (10, 20):   # the two list lengths from README decision 2
    i = int(round(top / 100 * n)) - 1
    ax.scatter(x[i], model_curve[i], color="C0", zorder=3)
    ax.annotate(f"top {top}%: {model_curve[i]:.1f}%", (x[i], model_curve[i]),
                textcoords="offset points", xytext=(8, 8), fontsize=9)
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)
ax.set_xlabel("Houses contacted (% of the ranking)")
ax.set_ylabel("Contacted houses really above 450 (%)")
ax.set_title(f"Hit rate by list length, {n} test houses")
ax.legend(loc="upper right", fontsize=9)
if TARGET_GROUP_HOUSES is not None:   # second axis: what each list length means in owners
    owners = ax.secondary_xaxis("top", functions=(lambda p: p / 100 * TARGET_GROUP_HOUSES,
                                                  lambda o: o / TARGET_GROUP_HOUSES * 100))
    owners.set_xlabel("Owners contacted in the target group")
fig.tight_layout()
fig.savefig(FIGURES / "evaluation_hit_rate_curve.png", dpi=150)
plt.show()

# the same curve as a table, for the report
rows = []
for top in (5, 10, 20, 30, 50, 100):
    k = int(round(top / 100 * n))
    rows.append({
        "list length (%)": top,
        "test houses on the list": k,
        "of which above 450": int(round(model_curve[k - 1] / 100 * k)),
        "hit rate model (%)": round(model_curve[k - 1], 1),
        "hit rate expert rule (%)": round(rule_curve[k - 1], 1),
        "random (%)": round(base_rate * 100, 1),
        "owners in target group": None if TARGET_GROUP_HOUSES is None else int(round(top / 100 * TARGET_GROUP_HOUSES)),
    })
hit_rates = pd.DataFrame(rows)
hit_rates.to_csv(TABLES / "evaluation_hit_rate_curve.csv", index=False)
hit_rates

**Result (hit-rate curve).** The model is above random choice at every list length. Against the expert rule the picture
depends on the length of the list: up to about a quarter of the ranking both are close (the rule is ahead at 10%, 60.0%
vs. 55.0%), between about 25% and 60% the model is clearly ahead (30%: 62.1% vs. 53.4%; 50%: 52.0% vs. 46.9%). The first
houses of the list are all hits; the dip around 7% comes from very few houses and is noise. In the target group of 21,377
undeclared houses (task 13), the top 10% means about 2,138 owners and the top 20% about 4,275 owners.